# 🧪 Technique Validation - Week 1 Baseline

**Purpose:** Validates all 8 IRS techniques used in SkillUp by running lightweight smoke tests against each module. In Week 1, the goal is "runs without crashing" — actual pass/fail scoring against success criteria happens in Weeks 2–3.

## Techniques Validated

1. **Semantic Similarity** (Stage 1/2 — Sentence-BERT cosine similarity)
2. **Knowledge Graph Queries** (Stage 2 — Neo4j role→skill traversal)
3. **NER Precision** (Data Pipeline — spaCy entity extraction)
4. **CSP (OR-Tools)** (Stage 3 — constraint satisfaction filtering)
5. **CBR (k-NN)** (Stage 3 — case-based retrieval)
6. **Fuzzy Logic** (Stage 3 — near-miss boundary handling)
7. **Competing Experts** (Stage 2 — JD demand vs peer CV arbitration)
8. **RAG Groundedness** (RAG Engine — attribution of explanations)

## Output

`evaluation/results/baseline_YYYYMMDD.json`

## Databricks Tables

* **Courses**: `workspace.default.my_skills_future_course_directory`
* **JDs**: `workspace.default.job_description`
* **Peers**: `workspace.default.resume_dataset_1200`
* **KG output**: `workspace.default.knowledge_graph_output`
* **Gap log**: `skillsup.gap_analysis.user_analysis_log`

---

In [0]:
# Install dependencies with python-constraint (no numpy conflict)
# Force downgrade numpy from 2.1.3 to 1.26.4
%pip install 'numpy==1.26.4' --force-reinstall --no-deps --quiet

# Install typing-extensions (required for spacy)
%pip install 'typing-extensions>=4.12.0' --upgrade --quiet

# Install ML packages (require numpy < 2.0)
%pip install pytest pytest-mock neo4j networkx pandas scikit-learn 'sentence-transformers>=3.0.0' 'spacy>=3.7.0' python-constraint openai --quiet

# Download spacy model
import subprocess
import sys

# Verify numpy version
import numpy as np
print(f"🔍 numpy version: {np.__version__}")
if np.__version__.startswith('1.26'):
    print("✅ numpy successfully downgraded to 1.26.x")
else:
    print(f"⚠️ numpy is {np.__version__} (expected 1.26.x)")

# Download spacy model
result = subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], capture_output=True, text=True)
if result.returncode == 0:
    print("✅ All dependencies installed successfully")
    print("✅ python-constraint: CSP solver with no numpy dependency")
    print("✅ Spacy model en_core_web_sm downloaded")
else:
    print("✅ Dependencies installed")
    print("⚠️ Spacy model download may have issues (will retry during validation if needed)")

In [0]:
%restart_python

### ✅ Dependency Configuration

**Solution:** Using numpy 1.26.x with python-constraint CSP solver to avoid dependency conflicts.

**Configuration:**
* **numpy 1.26.x**: Compatible with sentence-transformers, spacy, and databricks-connect
* **python-constraint**: Pure Python CSP solver (no numpy dependency, no conflicts)
* **Alternative to OR-Tools**: OR-Tools requires numpy>=2.0, which conflicts with ML libraries

**Result:** All 8 techniques can run in a single environment without dependency conflicts.

**Future Migration:**
* Monitor sentence-transformers/spacy numpy 2.0 compatibility (expected Q3 2026)
* Can switch back to OR-Tools once ecosystem migrates to numpy 2.0

In [0]:
import json
import time
import logging
import sys
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, Optional

# Environment detection - import dbutils for Serverless
try:
    from databricks.sdk.runtime import dbutils
    IN_DATABRICKS = True
    print("✅ Running in Databricks Serverless environment")
except ImportError:
    try:
        dbutils  # noqa - fallback for classic compute
        IN_DATABRICKS = True
        print("✅ Running in Databricks classic compute environment")
    except NameError:
        dbutils = None
        IN_DATABRICKS = False
        print("⚠️  Running outside Databricks — some features may not work")

# For Databricks, always use real mode (not mock)
MOCK_MODE = False

print(f"Environment: {'Databricks' if IN_DATABRICKS else 'Local'}")
print(f"Mock mode: {MOCK_MODE}")

In [0]:
import os

# ============================================================================
# PATH CONFIGURATION (Standard for all notebooks)
# ============================================================================

# Dynamic REPO_ROOT detection (works for any user)
try:
    # Try spark.conf first (works on Serverless)
    notebook_path = spark.conf.get("spark.databricks.notebook.path")
except:
    # Fallback to dbutils for classic compute
    try:
        notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    except:
        # Last resort - use current working directory
        notebook_path = str(Path.cwd())
        print("⚠️  Could not detect notebook path, using current directory")

# Convert notebook path to workspace path and derive repo root
# notebook_path is like: /Users/{username}/skillup/evaluation/notebooks/NotebookName
if notebook_path.startswith("/"):
    workspace_path = Path("/Workspace") / notebook_path.lstrip("/")
else:
    workspace_path = Path(notebook_path)

# Navigate up from notebooks -> evaluation -> skillup (repo root)
REPO_ROOT = workspace_path.parent.parent.parent if "notebooks" in str(workspace_path) else workspace_path

# Source data directory (version controlled)
DATA_DIR = REPO_ROOT / "data"

# Persistent artifact storage (Volumes - shared, not in git)
EVAL_ARTIFACTS = Path("/Volumes/workspace/default/iss-scratchpad/evaluation")
DATA_ARTIFACTS = Path("/Volumes/workspace/default/iss-scratchpad/data")

# Ensure artifact directories exist
EVAL_ARTIFACTS.mkdir(parents=True, exist_ok=True)
DATA_ARTIFACTS.mkdir(parents=True, exist_ok=True)

# Add skillup to Python path for imports
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"📁 REPO_ROOT: {REPO_ROOT}")
print(f"📁 DATA_DIR (source): {DATA_DIR}")
print(f"📦 EVAL_ARTIFACTS: {EVAL_ARTIFACTS}")
print(f"📦 DATA_ARTIFACTS: {DATA_ARTIFACTS}")

# ============================================================================
# DATA FILE PATHS
# ============================================================================

# Data file paths (in repo for version control)
SKILL_MAPPINGS_PATH = DATA_DIR / "skill_mappings_gold.json"
GOLD_CVS_PATH = DATA_DIR / "gold_standard_cvs.json"
GOLD_JDS_PATH = DATA_DIR / "gold_standard_jds.json"
TEST_PROFILES_PATH = DATA_DIR / "test_profiles.json"

# ============================================================================
# LOGGING SETUP
# ============================================================================

# Setup logging
log_file = EVAL_ARTIFACTS / f"validation_{datetime.now().strftime('%Y%m%d')}.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(log_file),
    ],
)
logger = logging.getLogger("technique_validation")
logger.info(f"✅ Logging to {log_file}")

# ============================================================================
# DATABRICKS SECRETS - Neo4j and OpenAI Configuration
# ============================================================================

if IN_DATABRICKS and dbutils is not None:
    try:
        # Retrieve Neo4j credentials from skillup scope
        # Note: Secret key is NEO4J_URL (not NEO4J_URI)
        neo4j_url = dbutils.secrets.get(scope="skillup", key="NEO4J_URL")
        neo4j_user = dbutils.secrets.get(scope="skillup", key="NEO4J_USER")
        neo4j_password = dbutils.secrets.get(scope="skillup", key="NEO4J_PASSWORD")
        neo4j_database = dbutils.secrets.get(scope="skillup", key="NEO4J_DATABASE")
        
        # Set as environment variables for knowledgegraph module
        # Map NEO4J_URL to NEO4J_URI for compatibility with module expectations
        os.environ["NEO4J_URI"] = neo4j_url
        os.environ["NEO4J_URL"] = neo4j_url
        os.environ["NEO4J_USER"] = neo4j_user
        os.environ["NEO4J_PASSWORD"] = neo4j_password
        os.environ["NEO4J_DATABASE"] = neo4j_database
        
        print("✅ Neo4j credentials configured from Databricks Secrets (skillup scope)")
        logger.info("Neo4j credentials loaded successfully")
    except Exception as e:
        print(f"⚠️  Neo4j credentials not found in Databricks Secrets: {e}")
        logger.warning(f"Neo4j credentials not configured: {e}")
    
    try:
        # Retrieve OpenAI API key from my-secrets scope
        openai_api_key = dbutils.secrets.get(scope="my-secrets", key="openai-api-key01")
        os.environ["OPENAI_API_KEY"] = openai_api_key
        
        print("✅ OpenAI API key configured from Databricks Secrets (my-secrets scope)")
        logger.info("OpenAI API key loaded successfully")
    except Exception as e:
        print(f"⚠️  OpenAI API key not found in Databricks Secrets: {e}")
        logger.warning(f"OpenAI API key not configured: {e}")
else:
    print("⚠️  Not in Databricks - Neo4j and OpenAI credentials must be configured manually")

In [0]:
# Result collector
results: Dict[str, Any] = {
    "snapshot_date": datetime.now().strftime("%Y-%m-%d"),
    "environment": "databricks" if IN_DATABRICKS else "local",
    "mock_mode": MOCK_MODE,
    "techniques": {},
    "latency": {
        "e2e_seconds": None,
        "skill_gap_seconds": None,
        "course_recommendation_seconds": None,
    },
    "notes": [],
}


def display_results(technique: str, details: Dict):
    """Display technique validation results in notebook."""
    status = details.get("status", "unknown")
    icon = "✅" if status == "runs" else "⚠️" if status == "partial" else "❌"
    
    print(f"\n{icon} {technique.upper().replace('_', ' ')} - Status: {status}")
    print("-" * 60)
    
    # Display key metrics based on technique
    for key, value in details.items():
        if key == "status":
            continue
        if isinstance(value, (list, dict)) and len(str(value)) > 200:
            print(f"  {key}: [truncated - see logs for full output]")
        else:
            print(f"  {key}: {value}")
    print()


def record(technique: str, status: str, details: Dict):
    """Record a technique result and display it."""
    results["techniques"][technique] = {"status": status, **details}
    icon = "✅" if status == "runs" else "⚠️" if status == "partial" else "❌"
    logger.info(f"{icon} [{technique}] status={status} | {details}")
    
    # Display results in notebook
    display_results(technique, {"status": status, **details})


def load_json(path: Path) -> Any:
    """Load a JSON file, returning None on failure."""
    try:
        with open(path) as f:
            return json.load(f)
    except Exception as e:
        logger.error(f"Failed to load {path}: {e}")
        print(f"❌ Failed to load {path}: {e}")
        return None


print("✅ Helper functions loaded")

## Technique 1: Semantic Similarity

**Goal:** Validate Sentence-BERT cosine similarity for skill matching

**Success Criteria:** ≥ 80% mean similarity score on skill mappings (Week 3)

**Week 1 Target:** Runs without crashing, logs sample scores

In [0]:
def validate_semantic_similarity():
    logger.info("=" * 60)
    logger.info("TECHNIQUE 1: Semantic Similarity")
    logger.info("=" * 60)

    # Load 5 sample skill pairs from gold mappings
    mappings = load_json(SKILL_MAPPINGS_PATH)
    if not mappings:
        record("semantic_similarity", "error", {"error": "Could not load skill_mappings_gold.json"})
        return

    sample_pairs = [(m["raw"], m["canonical"]) for m in mappings[:5]]
    logger.info(f"Sample pairs: {sample_pairs}")

    try:
        if MOCK_MODE:
            # Stub: return fixed scores
            scores = [0.92, 0.88, 0.95, 0.76, 0.83]
            logger.info("(MOCK) Cosine similarity scores: %s", scores)
            record("semantic_similarity", "runs", {
                "sample_scores": scores,
                "mean_score": round(sum(scores) / len(scores), 3),
                "note": "mock mode",
            })
        else:
            from sentence_transformers import SentenceTransformer, util
            model = SentenceTransformer("all-MiniLM-L6-v2")
            scores = []
            for raw, canonical in sample_pairs:
                emb1 = model.encode(raw, convert_to_tensor=True)
                emb2 = model.encode(canonical, convert_to_tensor=True)
                score = float(util.cos_sim(emb1, emb2))
                scores.append(round(score, 4))
                logger.info(f"  '{raw}' ↔ '{canonical}': {score:.4f}")
            record("semantic_similarity", "runs", {
                "sample_scores": scores,
                "mean_score": round(sum(scores) / len(scores), 3),
                "success_threshold": 0.80,
                "note": "Week 1: smoke test only — threshold validation in Week 3",
            })
    except Exception as e:
        logger.error(f"Semantic similarity failed: {e}")
        record("semantic_similarity", "error", {"error": str(e)})


print("✅ validate_semantic_similarity() defined")

## Technique 2: Knowledge Graph Queries

**Goal:** Validate Neo4j role→skill traversal

**Success Criteria:** ≥ 85% query correctness (Week 3)

**Week 1 Target:** Runs without crashing, retrieves skills for 5 IT roles

In [0]:
def validate_knowledge_graph():
    logger.info("=" * 60)
    logger.info("TECHNIQUE 2: Knowledge Graph Queries")
    logger.info("=" * 60)

    # 5 IT role queries drawn from gold_standard_jds.json
    it_roles = [
        "Data Scientist",
        "Software Engineer",
        "Machine Learning Engineer",
        "DevOps Engineer",
        "Data Analyst",
    ]

    try:
        if MOCK_MODE:
            mock_results = {
                role: [{"Python": 50}, {"SQL": 40}, {"Docker": 30}]
                for role in it_roles
            }
            logger.info("(MOCK) KG query results: %s", list(mock_results.keys()))
            record("knowledge_graph", "runs", {
                "roles_queried": it_roles,
                "roles_returning_results": len(it_roles),
                "sample_result": mock_results["Data Scientist"],
                "note": "mock mode — no Neo4j connection",
            })
        else:
            from knowledgegraph.knowledgegraph import get_skills_from_job

            role_results = {}
            for role in it_roles:
                try:
                    skills = get_skills_from_job(role)
                    role_results[role] = skills or []
                    logger.info(f"  {role}: {len(role_results[role])} skills retrieved")
                except Exception as role_err:
                    logger.warning(f"  {role}: query failed — {role_err}")
                    role_results[role] = None

            roles_with_data = sum(1 for v in role_results.values() if v)
            record("knowledge_graph", "runs" if roles_with_data > 0 else "partial", {
                "roles_queried": it_roles,
                "roles_returning_results": roles_with_data,
                "sample_result": role_results.get("Data Scientist"),
                "success_threshold": "≥ 85% query correctness (Week 3)",
            })
    except Exception as e:
        logger.error(f"Knowledge graph validation failed: {e}")
        record("knowledge_graph", "error", {"error": str(e)})


print("✅ validate_knowledge_graph() defined")

## Technique 3: NER Precision

**Goal:** Validate spaCy entity extraction for skills

**Success Criteria:** ≥ 80% precision on skill entities (Week 3, with custom model)

**Week 1 Target:** Runs with baseline en_core_web_sm, extracts entities from 3 CVs

In [0]:
def validate_ner():
    logger.info("=" * 60)
    logger.info("TECHNIQUE 3: NER Precision (spaCy)")
    logger.info("=" * 60)

    # Load 3 CVs for extraction test
    cvs = load_json(GOLD_CVS_PATH)
    if not cvs:
        record("ner", "error", {"error": "Could not load gold_standard_cvs.json"})
        return

    sample_cvs = cvs[:3]

    try:
        if MOCK_MODE:
            mock_entities = {
                "CV001": ["Python", "Windows Administration", "Jira", "IT Support"],
                "CV002": ["SQL", "Excel", "Accounting", "Stakeholder Management"],
                "CV003": ["Python", "Machine Learning", "REST API", "PostgreSQL"],
            }
            logger.info("(MOCK) NER extracted entities: %s", mock_entities)
            record("ner", "runs", {
                "cvs_tested": [cv["cv_id"] for cv in sample_cvs],
                "sample_entities": mock_entities,
                "note": "mock mode",
            })
        else:
            import spacy

            # ⚠️ TODO (Week 2): Use the project's custom spaCy NER model once trained.
            # For Week 1, using baseline en_core_web_sm to verify pipeline wiring.
            try:
                nlp = spacy.load("en_core_web_sm")
            except OSError:
                logger.warning("en_core_web_sm not found — run: python -m spacy download en_core_web_sm")
                record("ner", "error", {"error": "spaCy model not installed"})
                return

            ner_results = {}
            for cv in sample_cvs:
                # Read the markdown CV file for richer text
                md_path = REPO_ROOT / cv.get("md_file", "")
                if md_path.exists():
                    text = md_path.read_text(encoding="utf-8")
                else:
                    # Fall back to skill list as text
                    text = ", ".join(cv.get("skills", []))

                doc = nlp(text)
                entities = [(ent.text, ent.label_) for ent in doc.ents]
                ner_results[cv["cv_id"]] = entities[:10]  # top 10
                logger.info(f"  {cv['cv_id']}: {len(entities)} entities found")

            record("ner", "runs", {
                "cvs_tested": [cv["cv_id"] for cv in sample_cvs],
                "sample_entities": {k: v[:3] for k, v in ner_results.items()},
                "note": "Week 1: using en_core_web_sm baseline. Custom NER model needed in Week 2.",
                "success_threshold": "≥ 80% precision on skill entities (Week 3, with custom model)",
            })
    except Exception as e:
        logger.error(f"NER validation failed: {e}")
        record("ner", "error", {"error": str(e)})


print("✅ validate_ner() defined")

## Technique 4: CSP (OR-Tools)

**Goal:** Validate constraint satisfaction for course selection

**Success Criteria:** ≥ 90% constraint satisfaction on test cases (Week 3)

**Week 1 Target:** Runs on 2 test profiles (S3, S7), checks feasibility

In [0]:
def validate_csp():
    logger.info("=" * 60)
    logger.info("TECHNIQUE 4: CSP (OR-Tools)")
    logger.info("=" * 60)

    # Load 2 test profiles (S3, S7) — one relaxed budget, one tight
    profiles_data = load_json(TEST_PROFILES_PATH)
    if not profiles_data:
        record("csp", "error", {"error": "Could not load test_profiles.json"})
        return

    uat = profiles_data.get("uat_scenarios", [])
    sample_profiles = [p for p in uat if p["profile_id"] in ("S3", "S7")]

    try:
        if MOCK_MODE:
            mock_results = [
                {"profile_id": "S3", "feasible": True, "courses_selected": 3, "total_cost": 1800},
                {"profile_id": "S7", "feasible": True, "courses_selected": 1, "total_cost": 300},
            ]
            logger.info("(MOCK) CSP results: %s", mock_results)
            record("csp", "runs", {
                "profiles_tested": ["S3", "S7"],
                "results": mock_results,
                "note": "mock mode",
            })
        else:
            from recommender.recommender import CourseRecommender

            # Load candidate courses from Databricks table
            try:
                courses_df = spark.table("workspace.default.my_skills_future_course_directory")
                # Convert to list of dicts for recommender
                candidate_courses = courses_df.limit(100).toPandas().to_dict('records')
                logger.info(f"Loaded {len(candidate_courses)} candidate courses")
            except Exception as ce:
                logger.warning(f"Could not load courses from table: {ce}")
                # Use empty list for smoke test
                candidate_courses = []

            csp_results = []
            for profile in sample_profiles:
                try:
                    recommender = CourseRecommender()
                    # Simplified skill gap input for smoke test
                    result = recommender.recommend(
                        user_profile={
                            "skills": ["Python", "SQL"],
                            "target_role": profile["target_role"],
                            "budget": profile["budget_sgd"],
                            "weekly_hours": profile["weekly_hours_available"],
                            "modality": profile["modality"],
                        },
                        skill_gaps=[],  # Simplified for smoke test
                        candidate_courses=candidate_courses,
                    )
                    csp_results.append({
                        "profile_id": profile["profile_id"],
                        "feasible": result is not None,
                        "note": "full result logged to evaluation/results/",
                    })
                    logger.info(f"  {profile['profile_id']}: feasible={result is not None}")
                except Exception as pe:
                    logger.warning(f"  {profile['profile_id']}: {pe}")
                    csp_results.append({"profile_id": profile["profile_id"], "error": str(pe)})

            record("csp", "runs", {
                "profiles_tested": [p["profile_id"] for p in sample_profiles],
                "results": csp_results,
                "success_threshold": "≥ 90% constraint satisfaction on test cases (Week 3)",
            })
    except Exception as e:
        logger.error(f"CSP validation failed: {e}")
        record("csp", "error", {"error": str(e)})


print("✅ validate_csp() defined")

## Technique 5: CBR (k-NN)

**Goal:** Validate case-based retrieval for peer matching

**Success Criteria:** τ ≥ 0.65 vs human-ranked ground truth (Week 3, n=10 cases)

**Week 1 Target:** Logs placeholder — CBR module needs to be extracted

In [0]:
def validate_cbr():
    logger.info("=" * 60)
    logger.info("TECHNIQUE 5: CBR (k-NN)")
    logger.info("=" * 60)

    # Query 2 profiles and retrieve top-3 nearest cases
    query_profiles = [
        {"skills": ["Python", "SQL", "Scikit-learn"], "target": "Data Scientist"},
        {"skills": ["Java", "Spring Framework", "Docker"], "target": "DevOps Engineer"},
    ]

    try:
        if MOCK_MODE:
            mock_cases = [
                {
                    "query": query_profiles[0]["skills"],
                    "top_cases": [
                        {"case_id": "PEER001", "similarity": 0.91},
                        {"case_id": "PEER002", "similarity": 0.84},
                        {"case_id": "PEER003", "similarity": 0.79},
                    ],
                },
                {
                    "query": query_profiles[1]["skills"],
                    "top_cases": [
                        {"case_id": "PEER004", "similarity": 0.88},
                        {"case_id": "PEER005", "similarity": 0.81},
                    ],
                },
            ]
            logger.info("(MOCK) CBR results: %s", mock_cases)
            record("cbr", "runs", {
                "profiles_tested": 2,
                "results": mock_cases,
                "note": "mock mode",
            })
        else:
            # ⚠️ TODO (Week 2): Import CBR module once implemented in skillgap or recommender.
            # For now, log a placeholder result.
            logger.warning("CBR module not yet extracted as a standalone function.")
            logger.warning("ACTION NEEDED: Expose CBR retrieval as `run_cbr(skills, target_role, k=3)` in recommender.py")
            record("cbr", "partial", {
                "profiles_tested": 0,
                "note": (
                    "Week 1: CBR wiring incomplete. "
                    "Need to expose CBR retrieval function from recommender.py. "
                    "See ACTION NEEDED comment above."
                ),
                "success_threshold": "τ ≥ 0.65 vs human-ranked ground truth (Week 3, n=10 cases)",
            })
    except Exception as e:
        logger.error(f"CBR validation failed: {e}")
        record("cbr", "error", {"error": str(e)})


print("✅ validate_cbr() defined")

## Technique 6: Fuzzy Logic

**Goal:** Validate near-miss boundary handling

**Success Criteria:** ≥ 75% near-miss handling (Week 3, n=10 boundary cases)

**Week 1 Target:** Runs on 3 boundary cases, computes gap weights

In [0]:
def validate_fuzzy_logic():
    logger.info("=" * 60)
    logger.info("TECHNIQUE 6: Fuzzy Logic")
    logger.info("=" * 60)

    # 3 boundary skill pairs: user has partial proficiency
    boundary_cases = [
        {"skill": "Python", "user_proficiency": 0.4, "required_level": 0.6},
        {"skill": "Machine Learning", "user_proficiency": 0.3, "required_level": 0.7},
        {"skill": "SQL", "user_proficiency": 0.55, "required_level": 0.6},
    ]

    try:
        if MOCK_MODE:
            mock_scores = [
                {"skill": "Python", "gap_weight": 0.42, "is_near_miss": True},
                {"skill": "Machine Learning", "gap_weight": 0.65, "is_near_miss": False},
                {"skill": "SQL", "gap_weight": 0.12, "is_near_miss": True},
            ]
            logger.info("(MOCK) Fuzzy results: %s", mock_scores)
            record("fuzzy_logic", "runs", {
                "cases_tested": 3,
                "results": mock_scores,
                "near_miss_detected": 2,
                "note": "mock mode",
            })
        else:
            # Try to use fuzzy logic functionality if available
            try:
                from skillgap import skillgap
                
                # Check if fuzzy logic function is available
                if hasattr(skillgap, 'compute_gap_weight'):
                    fuzzy_results = []
                    for case in boundary_cases:
                        try:
                            gap_weight = skillgap.compute_gap_weight(
                                skill=case["skill"],
                                user_proficiency=case["user_proficiency"],
                                required_level=case["required_level"],
                            )
                            is_near_miss = 0.0 < gap_weight <= 0.3
                            fuzzy_results.append({
                                "skill": case["skill"],
                                "gap_weight": round(gap_weight, 4),
                                "is_near_miss": is_near_miss,
                            })
                            logger.info(f"  {case['skill']}: gap_weight={gap_weight:.4f}, near_miss={is_near_miss}")
                        except Exception as ce:
                            logger.warning(f"  {case['skill']}: {ce}")
                            fuzzy_results.append({"skill": case["skill"], "error": str(ce)})

                    near_miss_count = sum(1 for r in fuzzy_results if r.get("is_near_miss"))
                    record("fuzzy_logic", "runs", {
                        "cases_tested": len(boundary_cases),
                        "results": fuzzy_results,
                        "near_miss_detected": near_miss_count,
                        "success_threshold": "≥ 75% near-miss handling (Week 3, n=10 boundary cases)",
                    })
                else:
                    logger.warning(
                        "compute_gap_weight function not found in skillgap module. "
                        "ACTION NEEDED: Expose fuzzy gap computation as a public function."
                    )
                    record("fuzzy_logic", "partial", {
                        "cases_tested": 0,
                        "note": (
                            "Week 1: Fuzzy logic function not yet exposed in skillgap module. "
                            "Need to expose compute_gap_weight() function. "
                            "See ACTION NEEDED comment above."
                        ),
                        "success_threshold": "≥ 75% near-miss handling (Week 3, n=10 boundary cases)",
                    })
            except ImportError as ie:
                logger.warning(f"Could not import skillgap module: {ie}")
                record("fuzzy_logic", "partial", {
                    "cases_tested": 0,
                    "note": "Skillgap module not available",
                    "error": str(ie)
                })
    except Exception as e:
        logger.error(f"Fuzzy logic validation failed: {e}")
        record("fuzzy_logic", "error", {"error": str(e)})


print("✅ validate_fuzzy_logic() defined")

## Technique 7: Competing Experts

**Goal:** Validate JD demand vs peer CV arbitration

**Success Criteria:** ≥ 70% arbiter alignment with team consensus (Week 3, n=10 skill gaps)

**Week 1 Target:** Runs on 2 IT profiles, logs raw expert signals

In [0]:
def validate_competing_experts():
    logger.info("=" * 60)
    logger.info("TECHNIQUE 7: Competing Experts")
    logger.info("=" * 60)

    # Run 2 IT profiles through skill gap module and print raw expert signals
    test_inputs = [
        {
            "user_skills": ["Python", "SQL", "Pandas"],
            "target_role": "Data Scientist",
            "profile_id": "S3-simplified",
        },
        {
            "user_skills": ["Java", "Docker", "Git"],
            "target_role": "DevOps Engineer",
            "profile_id": "S4-simplified",
        },
    ]

    try:
        if MOCK_MODE:
            mock_outputs = [
                {
                    "profile_id": "S3-simplified",
                    "jd_demand_signal": {"Machine Learning": 0.85, "TensorFlow": 0.72},
                    "peer_cv_signal": {"Machine Learning": 0.88, "TensorFlow": 0.80},
                    "arbiter_output": {"Machine Learning": 0.87, "TensorFlow": 0.76},
                },
                {
                    "profile_id": "S4-simplified",
                    "jd_demand_signal": {"Kubernetes": 0.80, "Terraform": 0.70},
                    "peer_cv_signal": {"Kubernetes": 0.85, "Terraform": 0.65},
                    "arbiter_output": {"Kubernetes": 0.83, "Terraform": 0.68},
                },
            ]
            logger.info("(MOCK) Competing experts outputs: %s", [o["profile_id"] for o in mock_outputs])
            record("competing_experts", "runs", {
                "profiles_tested": 2,
                "results": mock_outputs,
                "note": "mock mode",
            })
        else:
            from skillgap.skillgap import find_skill_gaps

            expert_results = []
            for inp in test_inputs:
                try:
                    # Use function-based API
                    result = find_skill_gaps(
                        user_skills=inp["user_skills"],
                        target_role=inp["target_role"],
                    )
                    expert_results.append({
                        "profile_id": inp["profile_id"],
                        "gaps_found": len(result) if result else 0,
                    })
                    logger.info(f"  {inp['profile_id']}: gaps={len(result) if result else 0}")
                except Exception as ie:
                    logger.warning(f"  {inp['profile_id']}: {ie}")
                    expert_results.append({"profile_id": inp["profile_id"], "error": str(ie)})

            record("competing_experts", "runs" if expert_results else "partial", {
                "profiles_tested": len(test_inputs),
                "results": expert_results,
                "success_threshold": (
                    "≥ 70% arbiter alignment with team consensus labeling of 10 skill gaps (Week 3)"
                ),
                "note": (
                    "Week 1: logging raw expert signals only. "
                    "Consensus labeling ground truth to be created in Week 2."
                ),
            })
    except Exception as e:
        logger.error(f"Competing experts validation failed: {e}")
        record("competing_experts", "error", {"error": str(e)})


print("✅ validate_competing_experts() defined")

## Technique 8: RAG Groundedness

**Goal:** Validate RAG attribution of explanations to source documents

**Success Criteria:** ≥ 90% groundedness (Week 3, n=20 explanations)

**Week 1 Target:** Generates 2 explanations, checks for source attribution

In [0]:
def validate_rag():
    logger.info("=" * 60)
    logger.info("TECHNIQUE 8: RAG Groundedness")
    logger.info("=" * 60)

    # Generate 2 explanations and check retrieval source attribution
    test_cases = [
        {"target_role": "Data Scientist", "skills": ["Python", "SQL"]},
        {"target_role": "DevOps Engineer", "skills": ["Docker", "Linux/Unix"]},
    ]

    try:
        if MOCK_MODE:
            mock_explanations = [
                {
                    "case": "Data Scientist",
                    "explanation_snippet": "We recommend Machine Learning Fundamentals because...",
                    "retrieved_sources": ["course:SF002", "kg_node:Skill:MachineLearning"],
                    "grounded": True,
                },
                {
                    "case": "DevOps Engineer",
                    "explanation_snippet": "Based on 80% of DevOps JDs requiring Kubernetes...",
                    "retrieved_sources": ["kg_node:Skill:Kubernetes", "jd:JD009"],
                    "grounded": True,
                },
            ]
            logger.info("(MOCK) RAG groundedness outputs: %s", [r["case"] for r in mock_explanations])
            record("rag", "runs", {
                "cases_tested": 2,
                "explanations": mock_explanations,
                "grounded_count": 2,
                "note": "mock mode",
            })
        else:
            try:
                from app.llm_service import LLMService
                
                rag_results = []
                for case in test_cases:
                    try:
                        svc = LLMService()
                        result = svc.generate_explanation(
                            target_role=case["target_role"],
                            user_skills=case["skills"],
                        )
                        rag_results.append({
                            "target_role": case["target_role"],
                            "has_sources": bool(getattr(result, "sources", None)),
                            "explanation_length": len(str(result)),
                        })
                        logger.info(f"  {case['target_role']}: explanation generated, sources={bool(getattr(result, 'sources', None))}")
                    except Exception as re_:
                        logger.warning(f"  {case['target_role']}: {re_}")
                        rag_results.append({"target_role": case["target_role"], "error": str(re_)})

                record("rag", "runs" if rag_results else "partial", {
                    "cases_tested": len(test_cases),
                    "results": rag_results,
                    "success_threshold": "≥ 90% groundedness (Week 3, n=20 explanations, using rubric)",
                })
            except ImportError as ie:
                logger.warning(f"LLMService not available: {ie}")
                record("rag", "partial", {
                    "cases_tested": 0,
                    "note": "LLMService module not available - skipping real mode test",
                    "error": str(ie)
                })
    except Exception as e:
        logger.error(f"RAG validation failed: {e}")
        record("rag", "error", {"error": str(e)})


print("✅ validate_rag() defined")

## E2E Latency Measurement

**Goal:** Measure end-to-end pipeline latency

**Targets:**
* E2E total: < 15 seconds
* Skill gap: < 5 seconds
* Recommendation: < 7 seconds

**Test Profile:** CV003 (Software Developer → ML Engineer)

In [0]:
def measure_e2e_latency():
    logger.info("=" * 60)
    logger.info("E2E LATENCY MEASUREMENT")
    logger.info("=" * 60)

    if MOCK_MODE:
        results["latency"] = {
            "e2e_seconds": 12.4,
            "skill_gap_seconds": 3.1,
            "course_recommendation_seconds": 5.2,
            "note": "mock mode — not real measurements",
        }
        logger.info("(MOCK) Latency: %s", results["latency"])
        
        # Display mock results
        print("\n⏱️ E2E LATENCY MEASUREMENT")
        print("-" * 60)
        print(f"  E2E Total: {results['latency']['e2e_seconds']}s (target: <15s) ✅")
        print(f"  Skill Gap: {results['latency']['skill_gap_seconds']}s (target: <5s) ✅")
        print(f"  Recommendation: {results['latency']['course_recommendation_seconds']}s (target: <7s) ✅")
        print(f"  Note: {results['latency']['note']}")
        print()
        return

    try:
        # Use CV003 (Software Developer → ML Engineer) as the E2E test subject
        test_profile = {
            "user_skills": ["Python", "SQL", "REST API", "Git", "Scikit-learn"],
            "target_role": "Machine Learning Engineer",
            "budget": 2000,
            "weekly_hours": 10,
            "modality": "flexible",
        }

        e2e_start = time.time()

        # Stage 2: Skill gap (using function-based API)
        sg_start = time.time()
        from skillgap.skillgap import find_skill_gaps
        gap_result = find_skill_gaps(
            user_skills=test_profile["user_skills"],
            target_role=test_profile["target_role"],
        )
        sg_elapsed = time.time() - sg_start
        logger.info(f"Stage 2 (Skill Gap): {sg_elapsed:.2f}s")

        # Load candidate courses from Databricks table
        try:
            courses_df = spark.table("workspace.default.my_skills_future_course_directory")
            candidate_courses = courses_df.limit(100).toPandas().to_dict('records')
            logger.info(f"Loaded {len(candidate_courses)} candidate courses")
        except Exception as ce:
            logger.warning(f"Could not load courses: {ce}")
            candidate_courses = []

        # Stage 3: Course recommendation (using CourseRecommender class)
        rec_start = time.time()
        from recommender.recommender import CourseRecommender
        recommender = CourseRecommender()
        rec_result = recommender.recommend(
            user_profile={
                "skills": test_profile["user_skills"],
                "target_role": test_profile["target_role"],
                "budget": test_profile["budget"],
                "weekly_hours": test_profile["weekly_hours"],
                "modality": test_profile["modality"],
            },
            skill_gaps=gap_result if gap_result else [],
            candidate_courses=candidate_courses,
        )
        rec_elapsed = time.time() - rec_start
        logger.info(f"Stage 3 (Recommendation): {rec_elapsed:.2f}s")

        e2e_elapsed = time.time() - e2e_start
        logger.info(f"E2E Total: {e2e_elapsed:.2f}s")

        results["latency"] = {
            "e2e_seconds": round(e2e_elapsed, 2),
            "skill_gap_seconds": round(sg_elapsed, 2),
            "course_recommendation_seconds": round(rec_elapsed, 2),
            "targets": {"e2e": 15, "skill_gap": 5, "recommendation": 7},
            "meets_target": {
                "e2e": e2e_elapsed < 15,
                "skill_gap": sg_elapsed < 5,
                "recommendation": rec_elapsed < 7,
            },
        }
        
        # Display results
        print("\n⏱️ E2E LATENCY MEASUREMENT")
        print("-" * 60)
        e2e_icon = "✅" if e2e_elapsed < 15 else "❌"
        sg_icon = "✅" if sg_elapsed < 5 else "❌"
        rec_icon = "✅" if rec_elapsed < 7 else "❌"
        
        print(f"  E2E Total: {e2e_elapsed:.2f}s (target: <15s) {e2e_icon}")
        print(f"  Skill Gap: {sg_elapsed:.2f}s (target: <5s) {sg_icon}")
        print(f"  Recommendation: {rec_elapsed:.2f}s (target: <7s) {rec_icon}")
        print()

    except Exception as e:
        logger.error(f"E2E latency measurement failed: {e}")
        results["latency"]["error"] = str(e)
        results["notes"].append("E2E latency could not be measured — see error above")
        
        # Display error
        print("\n❌ E2E LATENCY MEASUREMENT FAILED")
        print("-" * 60)
        print(f"  Error: {str(e)}")
        print()


print("✅ measure_e2e_latency() defined")

## Run All Validations

Execute all 8 technique validations plus E2E latency measurement, then save results to JSON.

In [0]:
print("=" * 60)
print("SkillUp Technique Validation — Week 1 Baseline")
print(f"Date: {results['snapshot_date']}")
print(f"Environment: {results['environment']}")
print(f"Mock mode: {results['mock_mode']}")
print("=" * 60)
print()

# Run all validations
validate_semantic_similarity()
print()
validate_knowledge_graph()
print()
validate_ner()
print()
validate_csp()
print()
validate_cbr()
print()
validate_fuzzy_logic()
print()
validate_competing_experts()
print()
validate_rag()
print()
measure_e2e_latency()
print()

# Save results
output_file = EVAL_ARTIFACTS / f"baseline_{datetime.now().strftime('%Y%m%d')}.json"
with open(output_file, "w") as f:
    json.dump(results, f, indent=2)

logger.info("=" * 60)
logger.info(f"✅ Results saved to: {output_file}")

# Print summary
print("\n" + "=" * 60)
print("TECHNIQUE VALIDATION SUMMARY")
print("=" * 60)
for technique, data in results["techniques"].items():
    status = data.get("status", "unknown")
    icon = "✅" if status == "runs" else "⚠️" if status == "partial" else "❌"
    print(f"  {icon} {technique:<30} {status}")
print()
latency = results["latency"]
print(f"  E2E latency:               {latency.get('e2e_seconds', 'N/A')}s (target <15s)")
print(f"  Skill gap latency:         {latency.get('skill_gap_seconds', 'N/A')}s (target <5s)")
print(f"  Recommendation latency:    {latency.get('course_recommendation_seconds', 'N/A')}s (target <7s)")
print("=" * 60)
print(f"\n✅ Results saved to: {output_file}")
print("\n📊 Next steps:")
print(f"  1. Review baseline_{datetime.now().strftime('%Y%m%d')}.json")
print("  2. Address any 'partial' or 'error' statuses")
print("  3. Run UAT scenarios from test_profiles.json")
print("  4. Proceed to Week 2 development")